# Table 1 Analysis by Country

Compute Table 1 separately for USA and Argentina using `integrated_dream_data.csv` (already decoded). Each timepoint is treated as a study within country.
Metrics per timepoint: N (unique participants), age (median, SD using bin midpoints), gender %, education %.
For Argentina, if age/gender/education are missing at later timepoints, we carry forward the participant's earliest available values using `participant_id`.

In [26]:
from collections import Counter, defaultdict
import statistics
import csv

try:
    import pandas as pd
except ImportError:
    pd = None

CSV_PATH = "integrated_dream_data.csv"

AGE_MIDPOINTS = {
    "18-24": 21,
    "25-34": 29.5,
    "35-44": 39.5,
    "45-54": 49.5,
    "55-64": 59.5,
    "65-74": 69.5,
    "75-84": 79.5,
    "85 years": 85,
}

print("Imports ready. Adjust CSV_PATH if needed.")

Imports ready. Adjust CSV_PATH if needed.


In [27]:
# Load data
with open(CSV_PATH, newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows_raw = list(reader)
print(f"Rows: {len(rows_raw):,}")
print(f"Columns ({len(header)}): {header}")

with open(CSV_PATH, newline="") as f:
    rows = list(csv.DictReader(f))

Rows: 9,841
Columns (24): ['source_file', 'country', 'language', 'timepoint', 'participant_id', 'dream_text', 'dream_feelings', 'dream_talk', 'dream_write', 'dream_content', 'dream_frequency', 'dream_vivid', 'dream_bizarre', 'dream_emotional_tone', 'dream_intensity', 'gad_total', 'phq_total', 'age', 'education', 'student', 'gender', 'sleep_quality', 'sleep_hours', 'sleep_disturbed']


In [28]:
# Helpers


def classify_gender(raw):
    # Normalize gender strings/codes to coarse buckets.
    if raw is None:
        return None
    v = str(raw).strip().lower()
    if not v:
        return None
    if "female" in v:
        return "female"
    if "male" in v:
        return "male"
    if "non" in v and "binary" in v:
        return "non-binary"
    if "prefer" in v or "rather" in v:
        return "other"
    try:
        code = int(float(v))
        if code == 1:
            return "female"
        if code == 2:
            return "male"
        if code == 3:
            return "non-binary"
        if code == 4:
            return "other"
    except Exception:
        pass
    return "other"


def detect_timepoints(rows, country):
    tps = {
        row.get("timepoint")
        for row in rows
        if row.get("country") == country and row.get("timepoint")
    }
    return sorted(tps, key=lambda x: (int(x) if str(x).isdigit() else x))


def detect_education_categories(rows, country):
    cats = {
        (row.get("education") or "").strip()
        for row in rows
        if row.get("country") == country and row.get("education")
    }
    cats = {c for c in cats if c}
    return sorted(cats)


def build_profiles(rows, country):
    # Earliest available age/gender/education per participant for the given country.
    profiles = {}
    for row in rows:
        if row.get("country") != country:
            continue
        pid = row.get("participant_id")
        if not pid:
            continue
        prof = profiles.setdefault(pid, {})
        for key in ("age", "gender", "education"):
            if key not in prof:
                val = row.get(key)
                if val is not None and str(val).strip():
                    prof[key] = val
    return profiles


def compute_table(rows, country, timepoints=None, edu_cats=None):
    if timepoints is None:
        timepoints = detect_timepoints(rows, country)
    if edu_cats is None:
        edu_cats = detect_education_categories(rows, country)
    profiles = build_profiles(rows, country)
    results = {}
    for tp in timepoints:
        ids = set()
        ages = []
        genders = []
        edu_counts = Counter()
        for row in rows:
            if row.get("country") != country or row.get("timepoint") != tp:
                continue
            pid = row.get("participant_id")
            if pid in ids:
                continue
            ids.add(pid)

            prof = profiles.get(pid, {})
            age_val = row.get("age") or prof.get("age")
            gender_val = row.get("gender") or prof.get("gender")
            edu_val = row.get("education") or prof.get("education")

            age_label = (age_val or "").strip()
            if age_label in AGE_MIDPOINTS:
                ages.append(AGE_MIDPOINTS[age_label])
            gender = classify_gender(gender_val)
            if gender:
                genders.append(gender)
            edu = (edu_val or "").strip()
            if edu:
                edu_counts[edu] += 1

        N = len(ids)
        median_age = statistics.median(ages) if ages else None
        sd_age = statistics.pstdev(ages) if len(ages) > 1 else None
        gcounts = Counter(genders)
        total_g = sum(gcounts.values())
        pct = lambda k: gcounts[k] / total_g * 100 if total_g else None
        total_edu = sum(edu_counts.values())
        edu_pct = {
            cat: (edu_counts[cat] / total_edu * 100 if total_edu else None)
            for cat in edu_cats
        }
        results[tp] = {
            "N": N,
            "median_age": median_age,
            "sd_age": sd_age,
            "female_pct": pct("female"),
            "male_pct": pct("male"),
            "nb_pct": pct("non-binary"),
            "other_pct": pct("other"),
            "edu_pct": edu_pct,
        }
    return results

In [29]:
# Compute tables for both countries
countries = ["USA", "Argentina"]
tables = {}
meta = {}
for c in countries:
    tps = detect_timepoints(rows, c)
    edu = detect_education_categories(rows, c)
    tables[c] = compute_table(rows, c, tps, edu)
    meta[c] = {"timepoints": tps, "edu": edu}
    print(f"Country: {c} | timepoints: {tps} | education categories: {edu}")

Country: USA | timepoints: ['1', '2', '3', '4', '5'] | education categories: ['2 year degree', '4 year degree', 'Doctorate', 'High school graduate', 'Less than high school', 'Professional degree', 'Some college']
Country: Argentina | timepoints: ['1', '2', '3', '4', '5'] | education categories: ['4 year degree', 'Doctorate/PhD', 'High school graduate', 'Less than high school', "Master's degree/Certificate program", 'Professional degree', 'Some college']


In [31]:
# Formatting helpers


def fmt_age(rec):
    if rec["median_age"] is None or rec["sd_age"] is None:
        return ""
    return f"{rec['median_age']:.2f} ({rec['sd_age']:.2f})"


def fmt_pct(val):
    return f"{val:.2f}" if val is not None else ""


def build_table(country):
    tps = meta[country]["timepoints"]
    edu_categories = meta[country]["edu"]
    table = tables[country]

    headers = ["Metric"] + [
        f"Study {i} (timepoint {tp}, N={table[tp]['N']})"
        for i, tp in enumerate(tps, start=1)
    ]
    rows_out = []

    row_age = {"Metric": "Age (median, SD)"}
    row_female = {"Metric": "Female (%)"}
    row_male = {"Metric": "Male (%)"}
    row_nb = {"Metric": "Non-binary (%)"}
    row_other = {"Metric": "Other / Prefer not (%)"}

    for i, tp in enumerate(tps, start=1):
        col = headers[i]
        rec = table[tp]
        row_age[col] = fmt_age(rec)
        row_female[col] = fmt_pct(rec["female_pct"])
        row_male[col] = fmt_pct(rec["male_pct"])
        row_nb[col] = fmt_pct(rec["nb_pct"])
        row_other[col] = fmt_pct(rec["other_pct"])

    rows_out.extend([row_age, row_female, row_male, row_nb, row_other])

    for cat in edu_categories:
        r = {"Metric": f"Education: {cat}"}
        for i, tp in enumerate(tps, start=1):
            col = headers[i]
            r[col] = fmt_pct(table[tp]["edu_pct"].get(cat))
        rows_out.append(r)

    return headers, rows_out


for country in countries:
    headers, rows_out = build_table(country)
    print(f"Table 1 for {country}")
    if pd is not None:
        display(pd.DataFrame(rows_out))
    else:
        print(" | ".join(headers))
        print(" | ".join(["---"] * len(headers)))
        for r in rows_out:
            print(" | ".join(str(r.get(h, "")) for h in headers))

Table 1 for USA


,Metric,"Study 1 (timepoint 1, N=505)","Study 2 (timepoint 2, N=424)","Study 3 (timepoint 3, N=383)","Study 4 (timepoint 4, N=348)","Study 5 (timepoint 5, N=320)"
0,"Age (median, SD)",49.50 (17.61),49.50 (17.41),49.50 (17.52),49.50 (17.66),49.50 (17.39)
1,Female (%),51.68,49.17,49.35,51.59,53.75
2,Male (%),46.73,49.41,49.35,47.84,45.94
3,Non-binary (%),1.19,1.18,1.31,0.58,0.31
4,Other / Prefer not (%),0.40,0.24,0.00,0.00,0.00
5,Education: 2 year degree,12.28,12.06,12.27,11.24,12.19
6,Education: 4 year degree,33.86,33.57,34.73,34.29,34.38
7,Education: Doctorate,2.18,1.89,1.83,2.02,1.56
8,Education: High school graduate,9.90,10.64,10.97,10.66,11.25
9,Education: Less than high school,0.20,0.24,0.26,0.29,0.31


Table 1 for Argentina


,Metric,"Study 1 (timepoint 1, N=1054)","Study 2 (timepoint 2, N=877)","Study 3 (timepoint 3, N=755)","Study 4 (timepoint 4, N=665)","Study 5 (timepoint 5, N=615)"
0,"Age (median, SD)",29.50 (11.16),29.50 (11.18),29.50 (11.65),29.50 (11.58),29.50 (11.73)
1,Female (%),84.72,86.01,85.73,85.62,85.79
2,Male (%),12.90,11.63,11.84,12.25,12.31
3,Non-binary (%),1.42,1.61,1.28,1.31,0.87
4,Other / Prefer not (%),0.95,0.74,1.14,0.82,1.04
5,Education: 4 year degree,11.67,11.01,10.98,10.95,11.61
6,Education: Doctorate/PhD,6.26,5.82,6.70,7.35,7.11
7,Education: High school graduate,8.16,7.92,7.99,8.33,7.80
8,Education: Less than high school,0.19,0.12,0.00,0.00,0.00
9,Education: Master's degree/Certificate program,16.51,16.46,17.55,16.18,17.50


In [ ]:
# Persist separate CSVs
import csv

for country in countries:
    headers, rows_out = build_table(country)
    out_path = f"table1_summary_{country.lower()}.csv"
    with open(out_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows_out)
    print(f"Saved {out_path}")

## Argentina retention by participant
Counts of participants present at each timepoint, distinguishing those retained from the previous timepoint vs. new entrants. Uses `participant_id` (Argentina only).

In [ ]:
from collections import OrderedDict

arg_tps = meta["Argentina"]["timepoints"]
retention_rows = []
prev_ids = set()
seen_ids = set()
for tp in arg_tps:
    ids = {
        row["participant_id"]
        for row in rows
        if row.get("country") == "Argentina"
        and row.get("timepoint") == tp
        and row.get("participant_id")
    }
    staying = len(ids & prev_ids)
    new = len(ids - seen_ids)
    total = len(ids)
    retention_rows.append(
        {
            "timepoint": tp,
            "total_ids": total,
            "staying_from_prev": staying,
            "new_at_tp": new,
        }
    )
    prev_ids = ids
    seen_ids |= ids

if pd is not None:
    display(pd.DataFrame(retention_rows))
else:
    print(retention_rows)

Notes:
- Dataset is assumed already decoded; no Argentina mapping applied.
- Argentina age/gender/education are carried forward from the earliest available timepoint per participant when missing at later timepoints.
- Age uses bin midpoints; adjust `AGE_MIDPOINTS` if bins differ.
- If pandas is available, tables render as DataFrames; CSVs are always written.
- Update `CSV_PATH` if using an alternate integrated file (e.g., updated integration folder).